In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# ---------- 1. Load and clean data ----------
DATA_URL = "https://raw.githubusercontent.com/munirahsofeaMRT244075/agile-data-science-pma/refs/heads/main/data./telco.csv"  # e.g. https://raw.githubusercontent.com/.../data/telco.csv
df = pd.read_csv(DATA_URL)

df['TotalCharges'] = df['TotalCharges'].replace(' ', pd.NA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop_duplicates()
df = df.drop(columns=['customerID'])

# ---------- 2. Feature engineering ----------
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
categorical_cols = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

# ---------- 3. Train/test split + train the improved model ----------
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

improved_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced', random_state=42
)
improved_model.fit(X_train, y_train)
y_pred_improved = improved_model.predict(X_test)

# ---------- 4. Q5(a): Monitoring metrics ----------
# Metric 1: Model performance metric
current_f1 = f1_score(y_test, y_pred_improved)
print("Current model F1 score:", current_f1)

# Metric 2: Data quality metric
missing_pct = df.isnull().mean().mean() * 100
print(f"Average missing data percentage: {missing_pct:.2f}%")

Current model F1 score: 0.6362649294245385
Average missing data percentage: 0.00%


Drift Analysis

In [12]:
old_batch = df.iloc[:3500]
new_batch = df.iloc[3500:]

print("Old batch mean MonthlyCharges:", old_batch['MonthlyCharges'].mean())
print("New batch mean MonthlyCharges:", new_batch['MonthlyCharges'].mean())


Old batch mean MonthlyCharges: 65.01225714285714
New batch mean MonthlyCharges: 64.5141687835168
